In [ ]:
# Use a pipeline as a high-level helper
import torch
from transformers import pipeline

device = 0
if torch.cuda.is_available(): device = 0 # use cuda if available 
else: device = -1

pipe = pipeline(
      "automatic-speech-recognition", 
      model="Lingalingeswaran/whisper-small-sinhala",
      device=device
    )

def transcribe(audio):
    # Transcribe the audio file to text
    print(f'transcribing file: {audio}...')
    text = pipe(audio, return_timestamps=True)["text"]
    print('transcription finished.')
    return text

In [ ]:
!pip install pyngrok
!ngrok config add-authtoken $YOUR_AUTHTOKEN
'''
from google.colab import userdata
ngrok_key = userdata.get('ngrok_key')
!ngrok config add-authtoken $ngrok_key'''

from flask import Flask, request, jsonify
from pyngrok import ngrok
import shutil
import os

saveDir = '/content/audiofiles'
# ensure blank directory is made
try: shutil.rmtree(saveDir) # remove save directory
except Exception as e: print(f'Dir not found: {e}')

try: os.mkdir(saveDir) # create save directory
except Exception as e: print(f'Dir already exists: {e}')

app = Flask(__name__)

@app.route('/',methods=['GET','POST'])
def receive():
  data = request.files['file']
  print(f'------------------------------------\nFile Received: {data.filename}')

  data.save(f'{saveDir}/{data.filename}') # save to content/audiofiles

  text = transcribe(f'{saveDir}/{data.filename}') # extract trancription text
  print(f'{text[:30]}...') # preview of first 30 characters
  return jsonify({'reply':str(text)})

print(ngrok.connect(5000))
app.run(port=5000)